# Mantık ve Kümeler

**Modüller:** `pytop.logic`, `pytop.sets`  
**Konu:** Önerme mantığı, niceleyiciler, küme işlemleri, kuvvet kümesi, aile kavramı

---

Bu notebook beş bölümden oluşur:
1. **Konu** — Önerme nedir, küme teorisinin temelleri
2. **Teoremler** — De Morgan yasaları, Kartezyen çarpım, bölüm kümesi (tam ispat)
3. **API** — `Proposition`, mantık bağlaçları, niceleyiciler, küme kurucular
4. **Örnekler** — Dört senaryo: basit önermelerden indeksli ailelere
5. **Alıştırmalar** — Kodlama, teori ve karşılaştırma

---
## 1. Konu

### 1.1 Önerme Mantığı

Matematikte her ifade ya doğrudur ya yanlıştır. Bu ikili yapıyı formalize eden sistem **önerme mantığıdır**.

**Temel bağlaçlar:**

| Sembol | Ad | Doğruluk koşulu |
|--------|----|-----------------|
| $\neg p$ | değil (NOT) | $p$ yanlışsa |
| $p \wedge q$ | ve (AND) | her ikisi de doğruysa |
| $p \vee q$ | veya (OR) | en az biri doğruysa |
| $p \to q$ | gösterir (implies) | $p$ doğru, $q$ yanlış değilse |
| $p \leftrightarrow q$ | ancak ve ancak (iff) | aynı doğruluk değerine sahiplerse |

**Niceleyiciler** sonlu taşıyıcılar üzerinde:
- $\forall x \in S,\, P(x)$: $S$'nin her elemanı $P$'yi sağlar
- $\exists x \in S,\, P(x)$: $S$'de $P$'yi sağlayan en az bir eleman vardır
- $\exists! x \in S,\, P(x)$: $S$'de $P$'yi sağlayan tam bir eleman vardır

### 1.2 Küme Teorisi Temelleri

Bir **küme**, belirli nesnelerin koleksiyonudur. $x \in A$ ile $x$'in $A$'ya ait olduğunu gösteririz.

**Temel kavramlar:**
- **Boş küme** $\emptyset$: hiçbir eleman içermez
- **Alt küme** $A \subseteq B$: $A$'nın her elemanı $B$'dedir
- **Kuvvet kümesi** $\mathcal{P}(A)$: $A$'nın tüm alt kümelerinin kümesi; $|\mathcal{P}(A)| = 2^{|A|}$
- **Kartezyen çarpım** $A \times B = \{(a,b) : a \in A,\, b \in B\}$
- **İndeksli aile**: $\{A_i\}_{i \in I}$, bir indeks kümesi $I$ üzerinde tanımlı kümeler topluluğu

**Temel işlemler:**

| İşlem | Tanım |
|-------|-------|
| $A \cup B$ | $x \in A$ veya $x \in B$ |
| $A \cap B$ | $x \in A$ ve $x \in B$ |
| $A \setminus B$ | $x \in A$ ve $x \notin B$ |
| $A^c$ (evren $X$'e göre) | $X \setminus A$ |

---
## 2. Teoremler

### Teorem 1 — De Morgan Yasaları

**İfade:**  
$X$ evren, $A, B \subseteq X$ olsun:
1. $(A \cup B)^c = A^c \cap B^c$
2. $(A \cap B)^c = A^c \cup B^c$

**İspat (1):**  
$x \in (A \cup B)^c$  
$\iff x \notin A \cup B$  
$\iff x \notin A$ ve $x \notin B$  
$\iff x \in A^c$ ve $x \in B^c$  
$\iff x \in A^c \cap B^c$. $\square$

**İspat (2):** (1)'in aynalandırılmasıyla benzer şekilde gösterilir.

---

### Teorem 2 — Kuvvet Kümesinin Boyutu

**İfade:**  
$|A| = n$ ise $|\mathcal{P}(A)| = 2^n$.

**İspat (tümevarım):**  
*Taban:* $n=0$ için $A = \emptyset$, $\mathcal{P}(\emptyset) = \{\emptyset\}$, boyut $1 = 2^0$. ✓  
*Adım:* $|A| = n$ için doğru olduğunu varsay. $A' = A \cup \{a\}$ (yeni eleman $a$) için her $B \in \mathcal{P}(A)$'ya karşılık ya $B$ ya da $B \cup \{a\}$ geliyor — böylece $|\mathcal{P}(A')| = 2 \cdot 2^n = 2^{n+1}$. $\square$

---

### Teorem 3 — İndeksli Birleşim / Kesişim Dağılım Yasası

**İfade:**  
$A$ ve $\{B_i\}_{i \in I}$ kümeler ailesi için:
$$A \cap \left(\bigcup_{i \in I} B_i\right) = \bigcup_{i \in I}(A \cap B_i)$$

**İspat:**  
$x \in A \cap \bigcup B_i$  
$\iff x \in A$ ve $\exists i: x \in B_i$  
$\iff \exists i: x \in A \cap B_i$  
$\iff x \in \bigcup(A \cap B_i)$. $\square$

---
## 3. API

In [ ]:
from pytop.logic import (
    Proposition,
    negate, conjunction, disjunction, implies, iff,
    for_all, there_exists, unique_exists,
)
from pytop.sets import (
    make_set, empty_set, make_family,
    power_set, complement,
    set_union, set_intersection, set_difference,
    cartesian_product,
    indexed_union, indexed_intersection,
    equal_sets,
)
print("pytop yüklendi.")

### Mantık API'si

| Fonksiyon | Sözdizimi | Anlamı |
|-----------|-----------|--------|
| `Proposition(ad, değer)` | `Proposition("p", True)` | Adlandırılmış doğruluk değeri |
| `negate(p)` | `negate(p)` | $\neg p$ |
| `conjunction(p, q, ...)` | `conjunction(p, q)` | $p \wedge q \wedge \ldots$ |
| `disjunction(p, q, ...)` | `disjunction(p, q)` | $p \vee q \vee \ldots$ |
| `implies(p, q)` | `implies(p, q)` | $p \to q$ |
| `iff(p, q)` | `iff(p, q)` | $p \leftrightarrow q$ |
| `for_all(taşıyıcı, P)` | `for_all([1,2,3], lambda x: x>0)` | $\forall x\, P(x)$ |
| `there_exists(taşıyıcı, P)` | `there_exists([1,2,3], lambda x: x>2)` | $\exists x\, P(x)$ |
| `unique_exists(taşıyıcı, P)` | `unique_exists([1,2,3], lambda x: x==2)` | $\exists! x\, P(x)$ |

### Küme API'si

| Fonksiyon | Sözdizimi | Döndürür |
|-----------|-----------|----------|
| `make_set(*elemanlar)` | `make_set(1, 2, 3)` | `frozenset` |
| `empty_set()` | `empty_set()` | `frozenset()` |
| `make_family(*kümeler)` | `make_family({1,2}, {3})` | `frozenset[frozenset]` |
| `power_set(A)` | `power_set([1,2,3])` | tüm alt kümeler |
| `complement(A, X)` | `complement({1}, {1,2,3})` | $X \setminus A$ |
| `cartesian_product(A, B)` | `cartesian_product([1,2], ['a','b'])` | $A \times B$ |
| `indexed_union(aile)` | `indexed_union({'i': {1,2}, 'j': {3}})` | $\bigcup_i A_i$ |
| `indexed_intersection(aile)` | `indexed_intersection([{1,2}, {2,3}])` | $\bigcap_i A_i$ |

---
## 4. Örnekler

### Örnek 1 — Minimal: Temel Önerme İşlemleri

In [ ]:
p = Proposition("p", True)
q = Proposition("q", False)

print(f"p = {p}")
print(f"q = {q}")
print(f"¬p = {negate(p)}")
print(f"p ∧ q = {conjunction(p, q)}")
print(f"p ∨ q = {disjunction(p, q)}")
print(f"p → q = {implies(p, q)}")
print(f"p ↔ q = {iff(p, q)}")
print(f"q → p = {implies(q, p)}   # Yanlış öncül → her şey doğru")

### Örnek 2 — Orta Düzey: Niceleyiciler

In [ ]:
S = range(1, 6)   # {1, 2, 3, 4, 5}

# ∀x ∈ S: x > 0
print(f"∀x ∈ S, x > 0          → {for_all(S, lambda x: x > 0)}")

# ∀x ∈ S: x > 3
print(f"∀x ∈ S, x > 3          → {for_all(S, lambda x: x > 3)}")

# ∃x ∈ S: x² = 9
print(f"∃x ∈ S, x² = 9         → {there_exists(S, lambda x: x**2 == 9)}")

# ∃!x ∈ S: x² = 9
print(f"∃!x ∈ S, x² = 9        → {unique_exists(S, lambda x: x**2 == 9)}")

# ∃!x ∈ S: x çift
print(f"∃!x ∈ S, x çift        → {unique_exists(S, lambda x: x % 2 == 0)}")

### Örnek 3 — Gelişmiş: Küme İşlemleri ve De Morgan

In [ ]:
X = make_set(1, 2, 3, 4, 5)   # evren
A = make_set(1, 2, 3)
B = make_set(2, 3, 4)

union_AB    = set_union(A, B)
inter_AB    = set_intersection(A, B)
diff_AB     = set_difference(A, B)
comp_A      = complement(A, X)
comp_B      = complement(B, X)

print(f"A ∪ B     = {set(union_AB)}")
print(f"A ∩ B     = {set(inter_AB)}")
print(f"A \\ B     = {set(diff_AB)}")
print(f"Aᶜ        = {set(comp_A)}")
print(f"Bᶜ        = {set(comp_B)}")

# De Morgan doğrulaması: (A ∪ B)ᶜ = Aᶜ ∩ Bᶜ
lhs = complement(union_AB, X)   # union_AB zaten frozenset
rhs = set_intersection(comp_A, comp_B)
print(f"\n(A ∪ B)ᶜ        = {set(lhs)}")
print(f"Aᶜ ∩ Bᶜ          = {set(rhs)}")
print(f"De Morgan sağlandı: {equal_sets(lhs, rhs)}")

### Örnek 4 — Gerçekçi Senaryo: Kuvvet Kümesi, Kartezyen Çarpım ve İndeksli Aile

In [ ]:
# Kuvvet kümesi: 2^n = 8
base = make_set('a', 'b', 'c')
ps = power_set(base)
print(f"P({{'a','b','c'}}) = {len(ps)} eleman (2³ = {2**3})")
for s in sorted(ps, key=lambda x: (len(x), sorted(x))):
    print(f"  {set(s)}")

# Kartezyen çarpım
A2 = [1, 2]
B2 = ['x', 'y', 'z']
prod = cartesian_product(A2, B2)
print(f"\n{A2} × {B2} = {len(prod)} çift")
print(sorted(prod))

# İndeksli aile ve birleşim/kesişim
family = {'A1': {1, 2, 3}, 'A2': {2, 3, 4}, 'A3': {3, 4, 5}}
print(f"\n∪ Aᵢ = {indexed_union(family)}")
print(f"∩ Aᵢ = {indexed_intersection(family)}")

# make_family ile topoloji ailesi oluşturma
tau = make_family(set(), {1, 2, 3}, {1}, {2, 3})
print(f"\nTopoloji ailesi (make_family): {len(tau)} eleman")

---
## 5. Alıştırmalar

### Alıştırma 1 — Kodlama Görevi

$S = \{-2, -1, 0, 1, 2\}$ üzerinde şu soruları kodla:
- $\forall x \in S: x^2 \geq 0$
- $\exists x \in S: x^2 > 4$
- $\exists! x \in S: |x| = 0$

Sonra $A = \{-2, -1, 0\}$ ve $B = \{0, 1, 2\}$ için De Morgan'ı doğrula.

In [ ]:
S = range(-2, 3)
print(for_all(S, lambda x: x**2 >= 0))        # True bekleniyor
print(there_exists(S, lambda x: x**2 > 4))    # False bekleniyor
print(unique_exists(S, lambda x: abs(x) == 0)) # True bekleniyor

### Alıştırma 2 — Teori Sorusu

a) $|A| = 4$ için $|\mathcal{P}(\mathcal{P}(A))|$ kaçtır? Adım adım hesapla.  
b) $p \to q$ ile $\neg q \to \neg p$ (contrapositive) her zaman aynı doğruluk değerine mi sahiptir? `implies` ve `negate` kullanarak 4 durumu kontrol et.  
c) $A \times B$ ve $B \times A$ eşit midir? `equal_sets` ile kontrol et.

_Yanıtlarınızı buraya yazın:_

a) ...

b) ...

c) ...

### Alıştırma 3 — Karşılaştırma

$\bigcap_{i \in I} A_i$ ve $\bigcup_{i \in I} A_i$ arasındaki farkı örnekle göster.  
Özellikle $I = \emptyset$ (boş indeks) durumunda ne olur? `indexed_intersection` ve `indexed_union`'ı boş liste ile çağır.

In [ ]:
print("Boş indeks — birleşim: ", indexed_union([]))
print("Boş indeks — kesişim:  ", indexed_intersection([]))

# İndeksli aile ile dağılım yasasını doğrula: A ∩ (B ∪ C) = (A ∩ B) ∪ (A ∩ C)
A3 = make_set(1, 2, 3)
B3 = make_set(2, 4)
C3 = make_set(3, 4)
lhs2 = set_intersection(A3, set_union(B3, C3))
rhs2 = set_union(set_intersection(A3, B3), set_intersection(A3, C3))
print(f"Dağılım yasası sağlandı: {equal_sets(lhs2, rhs2)}")